In [12]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import requests
from pydantic import BaseModel

In [13]:
class cityname(BaseModel):
    city: str

In [14]:
load_dotenv()

True

In [15]:
model = ChatGroq(model="llama-3.3-70b-versatile")
model2 = model.with_structured_output(cityname) 


In [16]:
class WeatherState(TypedDict):
    user_query: str
    city: str
    weather_api_response: dict
    final_output: str

In [17]:
graph = StateGraph(WeatherState)

In [18]:
def extract_data(state: WeatherState) -> WeatherState:
    user_input = state['user_query']
    prompt = f'{user_input}'
    content = model2.invoke(prompt)
    state['city'] = content.city
    return state

In [19]:
def call_weather_api(state: WeatherState) -> WeatherState:
    city = state['city']
    api_key = '4753bbf1a1c715a1440c52412a447bcc'
    my_str = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric'
    response = requests.get(my_str)
    if response.status_code == 200:
        data = response.json()
    else:
        data = {'error': f'API returned {response.status_code}'}
    state['weather_api_response'] = data
    return state

In [20]:
def format_output(state: WeatherState) -> WeatherState:
    raw = state['weather_api_response']
    prompt = f'You have been given this raw data {raw} from the weather api key and now your work is that to make it a perfect and format sentence and format it'
    content = model.invoke(prompt).content
    state['final_output'] = content
    return state

In [21]:
graph.add_node('extract_data', extract_data)
graph.add_node('call_weather_api', call_weather_api)
graph.add_node('format_output', format_output)

graph.add_edge(START, 'extract_data')
graph.add_edge('extract_data', 'call_weather_api')
graph.add_edge('call_weather_api', 'format_output')
graph.add_edge('format_output', END)

workflow = graph.compile()

In [22]:
initial_input = {'user_query': 'What is the weather in London?'}
output = workflow.invoke(initial_input)
print(output['final_output'])

Based on the provided raw data from the weather API, here's a formatted and perfect sentence:

**Current Weather in London:**
The current temperature in London is 15.93°C, with a feels-like temperature of 15.33°C, and a humidity level of 67%. The sky is clear, with a visibility of 10 km and a gentle wind speed of 1.34 m/s. The minimum temperature is expected to be 14.2°C, while the maximum temperature is expected to be 17.21°C. The atmospheric pressure is 1024 hPa, and the cloud cover is 5%. The sunrise was at [insert time], and the sunset is expected to be at [insert time]. 

Here's a more detailed and formatted version:

* **Location:** London, GB
* **Current Temperature:** 15.93°C
* **Feels-like Temperature:** 15.33°C
* **Weather Condition:** Clear Sky
* **Humidity:** 67%
* **Visibility:** 10 km
* **Wind Speed:** 1.34 m/s
* **Wind Direction:** 50°
* **Gust:** 2.24 m/s
* **Atmospheric Pressure:** 1024 hPa
* **Cloud Cover:** 5%
* **Sunrise:** [insert time]
* **Sunset:** [insert time]
